# Avito bot detection challenge

## Подход к решению

Готовых признаков в данных нет, есть только лог событий. Поэтому основная
работа - придумать, что посчитать по событиям каждой куки. Я исходил из
трёх соображений.

1) Программа работает циклом, значит интервалы между
действиями у неё ровные. Она не читает объявление, а скачивает, значит на
карточке не задерживается. Обходит каталог по плану, значит идёт вширь по
категориям и вглубь по страницам выдачи. Отсюда временные признаки и
признаки охвата.

2) По каждой колонке смотрю: сколько разных значений,
всегда ли заполнена, что можно посчитать по группе, есть ли порядок.
Процедура даёт много признаков.

3) Первые два пункта описывают куку отдельно от
остальных, а такое поведение скрейпер имитирует легко: вставить случайные
паузы и полистать фотографии недорого. Сложнее скрыть другое — сервис
сбора данных обходит каталог по плану, и десятки его кук идут одним
маршрутом, тогда как живые люди расходятся. Значит стоит посчитать,
насколько популярны среди всех кук те объявления, которые открывала эта.

Блоки признаков добавляю по очереди и замеряю после каждого, чтобы видеть
вклад. Когда приросты станут меньше погрешности измерения, перейду с
одиночного временного сплита на кросс-валидацию.

Модель - LightGBM. Признаки разномасштабные, с пропусками и работают в сочетаниях: "свежая кука" сама по себе почти
ничего не значит, а "свежая кука, быстрые клики и популярные просмотренные
объявления" — уже сигнал. Деревья находят такие сочетания сами, линейной
модели их пришлось бы задавать руками. LightGBM обрабатывает пропуски сам,
масштабирование не нужно

Параметры подбираю в разделе 7. Забегая вперёд, их вклад близок к нулю:
качество здесь дают признаки

## 0. Импорты, настройки

In [187]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score
from metric import precision_at_recall, recall_at_fpr

SEED = 42
SEEDS = [42, 1, 7, 13, 21]

pd.set_option("display.width", 120)

## 1. Загрузка

In [189]:
DATA_COLS = ["cookie_created_at", "window_start_ts", "window_end_ts"]
train = pd.read_csv("data/train.csv", parse_dates=DATA_COLS)
test = pd.read_csv("data/test.csv", parse_dates=DATA_COLS)
events = pd.read_csv("data/events.csv.gz", parse_dates=["event_ts"])

print(train.shape, test.shape, events.shape)

(11091, 5) (4909, 4) (328905, 14)


## 2. EDA

### 2.1 Что в выборке

In [190]:
train.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [191]:
test.head()

,cookie_id,cookie_created_at,window_start_ts,window_end_ts
0,ck_315fb710a0e371e7,2026-02-20 08:52:31,2026-04-20,2026-04-21
1,ck_a76ee3b3e3e522fd,2026-01-19 13:42:15,2026-04-20,2026-04-21
2,ck_94c9a4d382689e82,2026-04-19 06:28:37,2026-04-20,2026-04-21
3,ck_8eaf9509ad9462a0,2026-01-19 17:18:06,2026-04-20,2026-04-21
4,ck_9a88a5a989cb5bc6,2025-11-09 17:25:13,2026-04-20,2026-04-21


In [192]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


In [193]:
print("доля ботов:", train["target"].mean())
print("окно для трейна:", train["window_start_ts"].min().date(), "-", train["window_start_ts"].max().date())
print("окно для теста:", test["window_start_ts"].min().date(), "-", test["window_start_ts"].max().date())
print("длина окна:", (train["window_end_ts"] - train["window_start_ts"]).unique())
print("пересечений cookie_id в трейне и тесте:", len(set(train["cookie_id"]) & set(test["cookie_id"])))
print("дубликатов cookie_id в трейне:", train["cookie_id"].duplicated().sum()) 

доля ботов: 0.0810567126498963
окно для трейна: 2026-04-06 - 2026-04-19
окно для теста: 2026-04-20 - 2026-04-26
длина окна: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]
пересечений cookie_id в трейне и тесте: 0
дубликатов cookie_id в трейне: 0


Отсюда видим, что тест идёт строго за трейном, пересечений нет

### 2.2 Эвенты за пределами окна

In [194]:
for name, meta in [("train", train), ("test", test)]:
    e = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
    after = e[e["event_ts"] >= e["window_end_ts"]]
    before = e[e["event_ts"] < e["window_start_ts"]]
    print(f"{name}:")
    print("\tсобытий всего:", len(e))
    print("\tиз них после окна:", len(after), f"({len(after)/len(e):.1%})")
    print("\tкук с событиями после окна:", after["cookie_id"].nunique(), "из", e["cookie_id"].nunique())
    print("\tдо окна:", len(before))

train:
	событий всего: 239215
	из них после окна: 40779 (17.0%)
	кук с событиями после окна: 5280 из 11091
	до окна: 0
test:
	событий всего: 89690
	из них после окна: 0 (0.0%)
	кук с событиями после окна: 0 из 4909
	до окна: 0


Видно, что 17% кук в трейне находится после требуемого окна. Значит, любой признак, обработанный без фильтра внесёт вклад в трейн, но ничего не покажет на тесте

### 2.3 Распределение капч

In [195]:
cap_count = (events[events["event_name"] == "captcha_shown"].groupby("cookie_id").size().reindex(train["cookie_id"]).fillna(0))
numbers = pd.cut(cap_count.values, [-1, 0, 1, 3, 10**6], labels=["0", "1", "2-3", ">3"])

print(pd.DataFrame({"число капч": numbers, "target": train["target"].values}).groupby("число капч", observed=True)["target"].agg(["mean", "size"]))

                mean   size
число капч                 
0           0.043540  10519
1           0.000000     52
2-3         0.000000     25
>3          0.890909    495


Больше 3 капч в среднем даёт 89% ботов. Однако все капчи лежат за границей окна (мы увидим это позже).
То есть, этот признак нам ничего не даст

### 2.4 Порядок строк и дубликаты

In [196]:
print("отсортирован по времени:", events["event_ts"].is_monotonic_increasing)
print("отсортирован по cookie_id:", events["cookie_id"].is_monotonic_increasing)
print("полных дублей строк", events.duplicated().sum())

dt_bad = events.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
dt_good = (events.sort_values(["cookie_id", "event_ts"]).groupby("cookie_id")["event_ts"].diff().dt.total_seconds())

print("отрицательных интервалов без сортировки:", (dt_bad < 0).sum())
print("отрицательных интервалов c сортировкой:", (dt_good < 0).sum())
print("медиана интервалов без сортировки:", dt_bad.median())
print("медиана интервалов c сортировкой:", dt_good.median())

отсортирован по времени: False
отсортирован по cookie_id: False
полных дублей строк 4863
отрицательных интервалов без сортировки: 156231
отрицательных интервалов c сортировкой: 0
медиана интервалов без сортировки: 0.0
медиана интервалов c сортировкой: 40.0


На перемешанном файлу половина интервалов отрицательные, медиана = 0. То есть, вся временная группа признаков превращается в шум

### 2.5 Обзор платформ

In [197]:
print(events["platform"].value_counts(), "\n")
print("разных написаний платформы у одной куки:")
print(events.groupby("cookie_id")["platform"].nunique().value_counts(), "\n")

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

разных написаний платформы у одной куки:
platform
3    7144
4    7100
2    1312
1     444
Name: count, dtype: int64 



Видим, что названия одни их тем же платформ написаны по-разному. Андроид в разных регистрах и iphone/ios как разные значения. У 7100 кук за сутки 4 разных описания, человек не меняет устройство 4 раза в день, значит регистр случайный

Также обратим внимание, что desktop это тоже, что и web:

In [198]:
print(events.groupby("platform").agg(n=("cookie_id", "size"), 
                                     ptr=("pointer_x", lambda s: s.notna().mean())).round(3))

              n    ptr
platform              
ANDROID   42462  0.000
Android   42254  0.000
IOS        4100  0.000
WEB       46476  0.581
Web       46365  0.588
android   42159  0.000
desktop   46866  0.587
iOS        4091  0.000
ios        4078  0.000
iphone     4103  0.000
web       45951  0.582


Доля событий с курсором у desktop и web совпадает до третьего знака (0.594 против 0.589-0.597). Значит, это одно и то же в этом случае

## 3. Фильтр окна и нормализация

In [199]:
def events_in_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """
    Оставляет эвенты внутри требуемого окна и сортирует сначала по cookie_id, затем по event_ts
    """
    ev = events.merge(meta[["cookie_id", "window_start_ts", "window_end_ts"]], on="cookie_id")
    ev = ev[(ev["event_ts"] >= ev["window_start_ts"]) & (ev["event_ts"] < ev["window_end_ts"])]
    return ev.sort_values(["cookie_id", "event_ts"])

def normalize_platform(ev: pd.DataFrame) -> pd.DataFrame:
    """
    Нормализует название платформы
    """
    return ev["platform"].str.lower().replace({"desktop": "web", "iphone": "ios"})

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
ev_all = pd.concat([ev_tr, ev_te], ignore_index=True)

print(len(ev_tr), len(ev_te))
print("капч внутри окна:", (ev_all["event_name"] == "captcha_shown").sum())
print("отрицательных интервалов:", (ev_tr.groupby("cookie_id")["event_ts"].diff().dt.total_seconds() < 0).sum())

198436 89690
капч внутри окна: 0
отрицательных интервалов: 0


Видно, что признак капчи бесполезен для задачи, потому что все капчи лежат вне требуемого окна

### Насколько таргеты отличаются (по признакам)

По каким признакам:
1. n - число событий куки внутри окна. Бот успевает за сутки сделать больше, чем человек
2. dt_med - медиана интервала между соседними событиями, в секундах. Бот делает всё с равномерными промежутками (задержками), человек читает и отвлекается
3. maxpage - до какой страницы выдачи дошла кука. Человек редко идёт дальше третьей
4. n_loc - сколько различных локаций затронуто. Человек будет искать в своём городе, бот затронет всё

Берём именно медиану в dt, потому что распределение интервалов скошено вправо

In [200]:
gap = ev_tr.groupby('cookie_id')['event_ts'].diff().dt.total_seconds().dropna()
print("Медиана:", gap.median())
print("Среднее:", round(gap.mean(), 1))

Медиана: 40.0
Среднее: 690.2


In [201]:
g = ev_tr.groupby("cookie_id")
probe = pd.DataFrame({
    "n": g.size(),
    "dt_med": g["event_ts"].diff().dt.total_seconds().groupby(ev_tr["cookie_id"]).median(),
    "maxpage": g["search_page"].max(),
    "n_loc": g["item_location"].nunique()
})
probe = train[["cookie_id", "target"]].merge(probe.reset_index(), on="cookie_id", how="left")
probe["cookie_id"] = (train["window_start_ts"] - train["cookie_created_at"]).dt.days.values
print(probe.groupby("target")[["n", "dt_med", "cookie_id", "maxpage", "n_loc"]].median())

           n  dt_med  cookie_id  maxpage  n_loc
target                                         
0       12.0    59.0       61.0      4.0    5.0
1       20.0    25.0       18.0      7.0    8.0


У ботов больше событий, короче интервалы, кука живёт меньше, страница больше, как и число локаций. Но показатели отличаются в 1.5-2 раза по медиане. Значит,
распределения сильно перекрывают друг друга: человек, который весь
вечер листает объявления, по любому из этих чисел будет выглядеть как бот.
Поставить порог по одному признаку и разделить классы корректно не получится

## 4. Признаки

Формирует матрицы признаков

In [202]:
EVENT_TYPES = ["item_view","search_results_view", "photo_swipe", "favorite_add",
               "seller_page_view", "contact_phone_show", "login",
               "contact_chat_open", "contact_message_sent"]

### 4.1 Статические агрегаты

Какие признаки берём:
1. n - сколько событий кука совершила внутри окна
2. n_item - число уникальных объявлений
3. n_cat - число уникальных категорий
4. n_loc - число уникальных локаций
5. n_q - число уникальных поисковых запросов
6. maxpage - самая дальняя страница поиска
7. meanpage - средний номер страницы
8. n_ua - число разных User-Agent у куки за сутки. У человека обычно один: браузер и устройство не меняется

In [203]:
def static_features(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """
    Собирает признаки по каждой куке: объём, разнообразие, состав действий
    """
    ev = ev.copy()
    ev["plat"] = normalize_platform(ev)
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        "n": g.size(),
        "n_item": g["item_id"].nunique(),
        "n_cat": g["item_category"].nunique(),
        "n_loc": g["item_location"].nunique(),
        "n_q": g["search_query"].nunique(),
        "maxpage": g["search_page"].max(),
        "meanpage": g["search_page"].mean(),
        "n_ua": g["user_agent"].nunique()
    })
    share = pd.crosstab(ev["cookie_id"], ev["event_name"], normalize="index") # доли типов событий
    for c in EVENT_TYPES:
        f["f_" + c] = share[c] if c in share else 0.0
    pshare = pd.crosstab(ev["cookie_id"], ev["plat"], normalize="index") # доли платформ
    for c in ["web", "android", "ios"]:
        f["plat_" + c] = pshare[c] if c in pshare else 0.0
    f["contact_rate"] = f.f_contact_phone_show + f.f_contact_chat_open + f.f_contact_message_sent # объединяем 3 вида контактов
    out = meta[["cookie_id"]].merge(f.reset_index(), on="cookie_id", how="left")
    out["cookie_age"] = ((meta["window_start_ts"].values - meta["cookie_created_at"].values) / np.timedelta64(1, "D"))
    return out

A_tr, A_te = static_features(ev_tr, train), static_features(ev_te, test)
Xtr, Xte = A_tr, A_te
print(Xtr.shape)

(11091, 23)


Посмотрим, что эти признаки дадут по сравнению с константой
В качестве модели возьмём LightGBM

In [204]:
y = train["target"].values
assert (Xtr["cookie_id"].values == train["cookie_id"].values).all()
is_valid = train["window_start_ts"].ge("2026-04-17").values # временной сплит

def quick(X: pd.DataFrame, cols: list[str] = None, name: str = "") -> None:
    """
    Обучаем модель и считаем метрику, затем выводим
    """
    cols = cols or [c for c in X.columns if c != "cookie_id"]
    m = lgb.LGBMClassifier(random_state=0, verbose=-1).fit(X[cols][~is_valid], y[~is_valid])
    p = m.predict_proba(X[cols][is_valid])[:, 1]
    s = precision_at_recall(y[is_valid], p)
    print(f"{name} | всего признаков: {len(cols)}, P@R70 = {s:.4f}")

print("константа | P@R70 =", round(y[is_valid].mean(), 4))
quick(Xtr, name="статические признаки")

константа | P@R70 = 0.082
статические признаки | всего признаков: 22, P@R70 = 0.3155


Дало прирост, формируем признаки дальше для улучшения метрики

### 4.2 Временные признаки

У человека есть неравномерные паузы между действиями: он задерживается на интересном, пролистывает скучное, отвлекается. У бота
интервалы почти всегда ровные, потому что цикл выполняется с одинаковой скоростью.
Посчитаем интервалы, их разброс и разделение активности на сессии

Берём признаки:

1. dt_p10, dt_med, dt_p90 - квантили интервала между соседними событиями. У бота пауза фиксирована, у человека половина действий быстрая, могут совпасть по медиане, но разойтись по краям, поэтому смотрим 3 квантиля
2. dt_min - самый короткий интервал
3. dt_lt2, dt_l5 - доля интервалов короче 2 и 5 секунд
4. rate - число событий в секунд за период активности
5. dt_std, dt_iqr - разброс интервалов
6. dt_cv - разброс относительно среднего, для честного сравнения быстрых и медленных кук
7. dt_mode_share - доля интервалов, попавших в самое частое значение. У скрипта с sleep(2) будет близко к единице
8. dt_reg - доля интервалов в пределах +-20% от медианы
9. span - активность от первого события до последнего
10. n_sess - число заходов. Разные сессии считается по разрыву между нимим в 30 минут
11. ev_per_sess - число событий за заход
12. sess_len_mean, sess_len_max - длина заходов (средняя и максимальная соответственно)
13. n_hour - в скольких разных часах суток кука активна
14. night - доля событий с полуночи до 6 утра

In [205]:
def time_features(ev):
    """
    Формирует временные признаки
    """
    ev = ev.copy()
    ev["dt"] = ev.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
    ev["hour"] = ev["event_ts"].dt.hour
    ev["new_sess"] = ev["dt"].isna() | (ev["dt"]> 1800) # разрыв 30 мин - новая сессия
    ev["sess"] = ev.groupby("cookie_id")["new_sess"].cumsum() # считаем номера сессий каждого события
    g = ev.groupby("cookie_id")
    f = pd.DataFrame({
        "dt_p10":  g["dt"].quantile(.10), "dt_med": g["dt"].median(), "dt_p90": g["dt"].quantile(.90),
        "dt_mean": g["dt"].mean(), "dt_std": g.dt.std(), "dt_min": g["dt"].min(),
        "span": (g["event_ts"].max() - g["event_ts"].min()).dt.total_seconds(),
        "n_hour": g["hour"].nunique(), "n_sess": g["sess"].max(),
    })
    f["dt_cv"]  = f["dt_std"] / (f["dt_mean"] + 1e-9)
    f["dt_iqr"] = g["dt"].quantile(.75) - g["dt"].quantile(.25)
    f["rate"] = g.size() / (f.span + 60)
    f["ev_per_sess"] = g.size() / f.n_sess
    f["dt_lt2"] = g["dt"].apply(lambda s: (s < 2).mean())
    f["dt_lt5"] = g["dt"].apply(lambda s: (s < 5).mean())
    f["night"] = g.hour.apply(lambda s: s.isin(range(6)).mean())
    f["dt_mode_share"] = g["dt"].apply(
        lambda s: s.round().value_counts(normalize=True).max() if s.notna().any() else np.nan)
    f["dt_reg"] = g["dt"].apply(
        lambda s: (np.abs(s - s.median()) < .2 * s.median()).mean() if s.notna().any() else np.nan)
    sl = ev.groupby(["cookie_id","sess"]).event_ts.agg(lambda s: (s.max() - s.min()).total_seconds())
    f["sess_len_mean"] = sl.groupby("cookie_id").mean()
    f["sess_len_max"]  = sl.groupby("cookie_id").max()
    return f.reset_index()

B_tr, B_te = time_features(ev_tr), time_features(ev_te)
Xtr = A_tr.merge(B_tr, on="cookie_id", how="left")
Xte = A_te.merge(B_te, on="cookie_id", how="left")
quick(Xtr, name="что было + временные признаки")

что было + временные признаки | всего признаков: 42, P@R70 = 0.5068


Видим прирост: с 0.3155 до 0.5068. Проверим, что будет, если оставить только временные признаки, без статических:

In [206]:
tcols = [c for c in time_features(ev_tr).columns if c != "cookie_id"]
quick(Xtr, tcols, "только временные признаки")

только временные признаки | всего признаков: 20, P@R70 = 0.2676


Без статических признаков метрика хуже. Значит, учитываем обе категории. Формируем ещё признаки

### 4.3 Признаки согласованности

Все предыдущие признаки описывают куку саму по себе: темп, её охват, её состав действий. Но бот умеет имитировать всё это.
Но он не может подделать согласованность со своими же куками.
Бот обходит каталог по плану: ему нужны определённые
категории, определённые локации, свежие объявления. Его куки идут
примерно одним маршрутом. У людей не так: один ищет одно в одном городе А, второй - другое в городе Б.

Выясним, насколько популярны среди всех остальных кук те объекты, которых касалась эта самая кука

Берём признаки:
cчитаем их по 3 столбцам: item_id, search_query, user_agent с разными префиксами (it, q, ua соответственно)

- item_id_ считаются по объявлениям
- q_ - по строкам поиска
- ua_ - по user agent

1. _pop_med - медиана по объявлениям куки от "сколько всего просмотров у этого объвления"
2. _pop_mean - среднее -//-
3. _pop_max - самое просматриваемое объявление из открытых кукой
4. _nck_med - медиана от "сколько разных кук видели это объявление"
5. _nck_mean - среднее -//-
6. _nck_max - объявление с наибольшим числом посетителей среди её просмотров
7. it_share_hot - доля объявлений куки, которые видели 8 и более кук

In [207]:
def popularity_features(ev_all: pd.DataFrame, ev_part: pd.DataFrame) -> pd.DataFrame:
    """
    Формирует признаки согласованности
    """
    f = pd.DataFrame(index=pd.Index(sorted(ev_part.cookie_id.unique()), name="cookie_id"))
    for col, pref in [("item_id","it"), ("search_query","q"), ("user_agent","ua")]:
        ref = ev_all.dropna(subset=[col])
        cnt = ref[col].value_counts() # число событий на объект
        nck = ref.groupby(col).cookie_id.nunique() # число различных кук на объект
        g = ev_part.dropna(subset=[col]).groupby("cookie_id")[col]
        f[pref + "_pop_med"] = g.apply(lambda s: cnt.reindex(s).median())
        f[pref + "_pop_mean"] = g.apply(lambda s: cnt.reindex(s).mean())
        f[pref + "_pop_max"] = g.apply(lambda s: cnt.reindex(s).max())
        f[pref + "_nck_med"] = g.apply(lambda s: nck.reindex(s).median())
        f[pref + "_nck_mean"]= g.apply(lambda s: nck.reindex(s).mean())
        f[pref + "_nck_max"] = g.apply(lambda s: nck.reindex(s).max())
        f[pref + "_share_hot"] = g.apply(lambda s: (nck.reindex(s) >= 8).mean())
    return f.reset_index()

C_tr, C_te = popularity_features(ev_all, ev_tr), popularity_features(ev_all, ev_te)
Xtr = Xtr.merge(C_tr, on="cookie_id", how="left")
Xte = Xte.merge(C_te, on="cookie_id", how="left")
quick(Xtr, name="что было + признаки согласованности")

что было + признаки согласованности | всего признаков: 63, P@R70 = 0.6957


Есть прирост: 0.5068 - 0.6957

In [208]:
print(Xtr.groupby(y)[["it_nck_mean", "it_pop_med", "ua_nck_mean"]].median())

   it_nck_mean  it_pop_med  ua_nck_mean
0        3.625         4.5        112.0
1        6.400         8.0        170.0


У ботов все три показателя примерно вдвое выше: они смотрят более многопросматриваемые
объявления и делят User-Agent с большим числом кук.

Разрыв тот же, что у dt_med или cookie_age. Но dt_med меряет, **как** кука себя ведёт, и обходится вставкой
случайных пауз. Эти признаки меряют, **куда** она ходит, — чтобы их сбить,
боту пришлось бы отказаться от плана обхода, то есть перестать делать
свою работу.

Формируем ещё признаки

### 4.4 Уточняющие признаки

Здесь проходим по уже использованным колонкам ещё раз и достаём упущенное прошлыми категориями признаков: 

1. Абсолютные числа. В статистических признаках (п. 4.1) события посчитаны долями, а доля не различает куку с дессятью эвентами и куку с 300-ми
2. Равномерность охвата. Мы считали, сколько у куки разных категорий, но не считали, как события распределились между ними
3. Порядок действий. Мы считали, сколько каких событий. Но не смотрели последовательность, в которой они шли

Берём признаки:

I. Абсолютные числа:
1. c_item_view, c_search_results_view, c_photo_swipe, c_favorite_add, c_seller_page_view, c_contact_phone_show, c_login, c_contact_chat_open, c_contact_message_sent - количество событий каждого типа в штуках
2. iv_per_srv - сколько карточек открыто на одну загрузку выдачи
3. item_rep - доля уникальных среди просмотренных объявлений; низкое значение означает, что кука ходит по кругу
4. q_rep - то же для поисковых запросов

II. Равномерность охвата:
1. ent_cat - энтропия по категориям
2. ent_loc - по локациям
3. ent_event - по типам событий
4. ent_hour - по часам суток
5. deep_page - доля событий поиска на 5-ой странице и дальше

III. Порядок действий:
1. n_trans - сколько разных пар «событие - следующее событие» встретилось
2. trans_top - доля самой частой такой пары
3. same_item_next - доля событий с тем же объявлением, что и в предыдущем
4. page_seq_inc - доля переходов, где номер страницы вырос ровно на 1
5. dt_med_item_view, dt_med_search_results_view, dt_med_photo_swipe — медиана интервала отдельно по каждому типу события. Общий dt_med смешивает быстрые технические события с медленным чтением карточки

IV. Остальное:
1. ptr_frac_web - доля событий с координатами курсора, считается только по web-событиям: на мобильных курсора нет физически, иначе признак станет копией plat_web
2. seller_pro - доля просмотров у профессиональных продавцов
3. ua_lib - флаг: в User-Agent есть curl, python, requests
4. ua_head - флаг HeadlessChrome

In [209]:
def extra_features(ev: pd.DataFrame) -> pd.DataFrame:
    """
    Формирует уточняющие признаки
    """
    ev = ev.copy()
    ev["plat"] = normalize_platform(ev)
    g = ev.groupby("cookie_id")
    f = pd.DataFrame(index=pd.Index(sorted(ev["cookie_id"].unique()), name="cookie_id"))
    cnt = pd.crosstab(ev["cookie_id"], ev["event_name"])
    for c in EVENT_TYPES:
        f["c_" + c] = cnt[c] if c in cnt else 0
    f["iv_per_srv"] = f["c_item_view"] / (f["c_search_results_view"] + 1)
    f["item_rep"] = g["item_id"].nunique() / (g["item_id"].count() + 1e-9)
    f["q_rep"] = g["search_query"].nunique() / (g["search_query"].count() + 1e-9)

    def ent(s: pd.Series) -> float:
        """
        Считает энтропию
        """
        p = s.value_counts(normalize=True).values
        return -(p * np.log(p + 1e-12)).sum()

    f["ent_event"] = g["event_name"].apply(ent)
    f["ent_cat"] = g["item_category"].apply(ent)
    f["ent_loc"] = g["item_location"].apply(ent)
    f["ent_hour"] = g["event_ts"].apply(lambda s: ent(s.dt.hour))

    ev["prev"] = g["event_name"].shift()
    tt = ev.dropna(subset=["prev"]).assign(tr=lambda d: d["prev"] + ">" + d["event_name"])
    f["n_trans"] = tt.groupby("cookie_id")["tr"].nunique()
    f["trans_top"] = tt.groupby("cookie_id")["tr"].apply(lambda s: s.value_counts(normalize=True).max())
    f["same_item_next"] = g["item_id"].apply(lambda s: (s == s.shift()).mean())

    f["deep_page"] = g["search_page"].apply(lambda s: (s >= 5).mean())
    pg = ev.dropna(subset=["search_page"]).groupby("cookie_id")["search_page"]
    f["page_seq_inc"] = pg.apply(lambda s: (s.diff() == 1).mean())
    w = ev[ev["plat"] == "web"] # курсор существует только на web
    f["ptr_frac_web"] = w.groupby("cookie_id")["pointer_x"].apply(lambda s: s.notna().mean())
    f["seller_pro"] = g["seller_type"].apply(lambda s: (s == "pro").mean())

    ev["ua_lib"] = ev["user_agent"].str.contains("curl|python|requests|Scrapy|Go-http|okhttp|wget", case=False)
    ev["ua_head"] = ev["user_agent"].str.contains("HeadlessChrome")
    f["ua_lib"] = ev.groupby("cookie_id")["ua_lib"].max().astype(int)
    f["ua_head"] = ev.groupby("cookie_id")["ua_head"].max().astype(int)

    for e_name in ["item_view", "search_results_view", "photo_swipe"]:
        s = ev[ev["event_name"] == e_name].copy()
        s["d"] = s.groupby("cookie_id")["event_ts"].diff().dt.total_seconds()
        f["dt_med_" + e_name] = s.groupby("cookie_id")["d"].median()

    return f.reset_index()


D_tr, D_te = extra_features(ev_tr), extra_features(ev_te)
Xtr = Xtr.merge(D_tr, on="cookie_id", how="left")
Xte = Xte.merge(D_te, on="cookie_id", how="left")
quick(Xtr, name="что было + уточняющие признаки")

что было + уточняющие признаки | всего признаков: 91, P@R70 = 0.7368


Есть прирост: 0.6957 - 0.7368

## 5. Схема валидации

Переходим на кросс-валидацию: каждая кука попадает в валидацию ровно один
раз за проход, метрика считается по всем 899 ботам, а разные сиды
перемешивают разбиение и гасят удачные и неудачные нарезки

In [210]:
PARAMS_HAND = dict(objective="binary", learning_rate=0.05, num_leaves=63,
                   min_child_samples=20, feature_fraction=0.8,
                   bagging_fraction=0.8, bagging_freq=1,
                   n_estimators=600, verbose=-1, n_jobs=-1)

def cv_oof(Xf, y, *, seeds=SEEDS, params=None):
    """
    Cчитает OOF-предсказания, усреднённые по нескольким разбиениям
    """
    params = params or PARAMS_HAND
    oof = np.zeros(len(y))
    for sd in seeds:
        p = dict(params); p["random_state"] = sd
        for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=sd).split(Xf, y):
            m = lgb.LGBMClassifier(**p).fit(Xf.iloc[tr_idx], y[tr_idx])
            oof[va_idx] += m.predict_proba(Xf.iloc[va_idx])[:, 1]
    return oof / len(seeds)

web = Xtr["plat_web"].values > 0.5

def report(name, oof):
    print(f"{name:14s} P@R70 {precision_at_recall(y, oof):.4f} | "
          f"PR {average_precision_score(y, oof):.4f} | "
          f"ROC {roc_auc_score(y, oof):.4f} | "
          f"R@FPR1% {recall_at_fpr(y, oof):.4f}")

## 6. Отбор: удаление ненужных признаков

Использовался 91 признак. Нужно проверить, есть ли среди них ненужные

Мешают признаки, которые почти не меняются от куки к куке, их надо убрать, чтобы разбиение в деревьях было осмысленное

Признаки согласованности считают, сколько кук затронуло тот же объект. Такой счётчик работает, только если объектов много: тогда одни редкие, другие популярные. Если объектов мало, каждый затронут сотнями кук и счётчик у всех одинаковый

In [213]:
for c in ["item_id", "search_query", "user_agent"]:
    print(f"{c} уникальных: {events[c].nunique()}")
print()
print(Xtr[[c for c in Xtr.columns if c.startswith("q_pop") or c.startswith("q_nck")
           or c == "q_share_hot"]].describe().T[["mean", "std", "min", "max"]].round(2))

item_id уникальных: 53856
search_query уникальных: 120
user_agent уникальных: 148

               mean    std    min    max
q_pop_med    749.36  31.61  660.0  839.0
q_pop_mean   749.57  28.22  660.0  839.0
q_pop_max    782.80  36.97  660.0  839.0
q_nck_med    519.58  15.45  460.0  564.0
q_nck_mean   519.28  14.16  460.0  564.0
q_nck_max    536.36  18.16  460.0  564.0
q_share_hot    1.00   0.00    1.0    1.0


Объявлений 54 тысячи на 16 тысяч кук — каждое видели три-четыре куки,
разброс есть. А запросов всего 120, то есть каждый вводили сотни кук

Видно по числам: q_nck_med у всех лежит между 460 и 564 при среднем 519.
q_share_hot вообще константа - 1.0 у всех, стандартное отклонение ноль,
потому что любой из 120 запросов вводили больше восьми человек. Для
сравнения, it_nck_mean меняется от 1 до 14.5

Проверяем удалением, обоими инструментами: грубым и точным

q_rep при этом оставляем, так как это доля уникальных запросов у самой куки,
обычный признак со стандартным отклонением 0.30, к признакам согласованности он
отношения не имеет

In [214]:
Q_DROP = ["q_pop_med", "q_pop_mean", "q_pop_max",
          "q_nck_med", "q_nck_mean", "q_nck_max", "q_share_hot"]
cols_all = [c for c in Xtr.columns if c != "cookie_id"]
cols = [c for c in cols_all if c not in Q_DROP]

print("временной сплит:")
quick(Xtr, cols_all, "  со всеми q_")
quick(Xtr, cols, "  без q_")

print("\nкросс-валидация:")
for nm, cc in [("  со всеми q_", cols_all), ("  без q_", cols)]:
    oof = cv_oof(Xtr[cc], y, seeds=(42, 1, 7))
    print(f"{nm:14s} признаков {len(cc):3d} | P@R70 {precision_at_recall(y, oof):.4f} "
          f"| PR-AUC {average_precision_score(y, oof):.4f}")

print("\nитого признаков:", len(cols))

временной сплит:
  со всеми q_ | всего признаков: 91, P@R70 = 0.7368
  без q_ | всего признаков: 84, P@R70 = 0.7044

кросс-валидация:
  со всеми q_  признаков  91 | P@R70 0.7905 | PR-AUC 0.7813
  без q_       признаков  84 | P@R70 0.8087 | PR-AUC 0.7854

итого признаков: 84


Два замера дают противоположный ответ. Временной сплит: 0.7368 -
0.7044, удаление выглядит ухудшением. Кросс-валидация: 0.7905 против
0.8087, удаление улучшает, и PR-AUC подтверждает: 0.7813 - 0.7854

Это одни и те же признаки. Разница в том,
по скольким ботам считается метрика: 160 против 899. Эффект в +0.018
временной сплит увидеть не способен, и знак у него оказался случайным

Верим кросс-валидации: прирост по PR-AUC +0.004 при её разбросе около
0.002. Плюс структурный аргумент: q_share_hot
буквально константа.

Убираем 7 признаков, остаётся 84

Дальше функцию quick использовать не будем, она свою роль выполнила на этапе сборки
признаков, когда приросты шли по +0.2

## 7. Подбор гиперпараметров

Порядок такой: сначала определяем число деревьев, потом измеряем шум метрики, и только потом запускаем поиск, иначе найденный прирост будет не с чем сравнивать 

### 7.1. Сколько деревьев нужно на самом деле

Ранняя остановка внутри фолдов отвечает на этот вопрос напрямую

In [215]:
p = {k: v for k, v in PARAMS_HAND.items() if k != "n_estimators"}
best_iters = []
for a, b in StratifiedKFold(5, shuffle=True, random_state=SEED).split(Xtr[cols], y):
    m = lgb.LGBMClassifier(**p, n_estimators=3000, random_state=SEED)
    m.fit(Xtr[cols].iloc[a], y[a],
          eval_X=Xtr[cols].iloc[b], eval_y=y[b],
          eval_metric="average_precision",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    best_iters.append(m.best_iteration_)
print("оптимальное число итераций по фолдам:", best_iters, "| медиана:", int(np.median(best_iters)))

оптимальное число итераций по фолдам: [62, 63, 80, 74, 67] | медиана: 67


Получаем оптимум около 67 итераций, а в PARAMS_HAND стоит 600. Это перебор: качество от этого почти не страдает, но время обучения растёт

Пространство поиска ниже строим вокруг найденного значения, с запасом:
150, 250, 400.

Оговорка по методу: ранняя остановка смотрит на тот же фолд, где потом
меряется качество, поэтому оценка слегка оптимистична. Для определения
порядка величины (70, а не 600) этого достаточно. Строго правильно было
бы выделить внутренний сплит

### 7.2 Шум измерения

Прежде чем сравнивать конфигурации, надо понять, какая разница между ними вообще значима. Для этого меряем одну и ту же конфигурацию на разных сидах: весь разброс, который получится - чистый шум

In [216]:
noise_pr = [average_precision_score(y, cv_oof(Xtr[cols], y, seeds=(s,))) for s in SEEDS]
noise_pa = [precision_at_recall(y, cv_oof(Xtr[cols], y, seeds=(s,))) for s in SEEDS]
SIGMA = float(np.std(noise_pr))
print("PR-AUC по сидам:", np.round(noise_pr, 4), "| sigma =", round(SIGMA, 4))
print("P@R70  по сидам:", np.round(noise_pa, 4), "| sigma =", round(float(np.std(noise_pa)), 4))

PR-AUC по сидам: [0.7811 0.7798 0.7815 0.782  0.7782] | sigma = 0.0014
P@R70  по сидам: [0.7995 0.7793 0.7985 0.7885 0.7905] | sigma = 0.0074


PR-AUC в пределах 0.0014, целевая метрика - 0.0074, то есть в 5
раз больше. Причина в дискретности: P@R70 равна отношению двух целых чисел
(сколько ботов поймали на сколько отмеченных кук), промежуточных значений
у неё не бывает, и она скачет

Поэтому конфигурации будем отбирать по PR-AUC, а P@R70 оставим для отчёта.
И запоминаем порог значимости: разницу меньше 0.003 по PR-AUC принимать
за улучшение нельзя

### 7.3 Случайный поиск

Подбирать гиперпараметры будем случайным поиском, а не полным перебором, так как иначе сетка перебора очень большая. Переберём 12 комбинаций

In [217]:
SPACE = dict(learning_rate=[.03, .05, .08],
             num_leaves=[31, 63, 127],
             min_child_samples=[10, 20, 40],
             feature_fraction=[.6, .8, 1.],
             bagging_fraction=[.7, .9],
             reg_lambda=[0, 1, 5],
             n_estimators=[150, 250, 400])

rs = np.random.RandomState(0)
results, seen = [], set()
for _ in range(12):
    while True:
        c = {k: v[rs.randint(len(v))] for k, v in SPACE.items()}
        key = tuple(sorted(c.items()))
        if key not in seen:
            seen.add(key); break
    pr = average_precision_score(y, cv_oof(Xtr[cols], y, seeds=(42, 1), params={**PARAMS_HAND, **c}))
    results.append((pr, c))
    print(f"{pr:.4f}  {c}")

results.sort(reverse=True, key=lambda r: r[0])
pr_hand = average_precision_score(y, cv_oof(Xtr[cols], y, seeds=(42, 1)))
print(f"\nручная конфигурация: {pr_hand:.4f}")
print(f"лучшая из поиска: {results[0][0]:.4f}")

0.7823  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'reg_lambda': 1, 'n_estimators': 400}
0.7732  {'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 10, 'feature_fraction': 0.6, 'bagging_fraction': 0.7, 'reg_lambda': 5, 'n_estimators': 250}
0.7784  {'learning_rate': 0.08, 'num_leaves': 127, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'reg_lambda': 1, 'n_estimators': 250}
0.7825  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 10, 'feature_fraction': 0.6, 'bagging_fraction': 0.9, 'reg_lambda': 1, 'n_estimators': 400}
0.7793  {'learning_rate': 0.03, 'num_leaves': 127, 'min_child_samples': 10, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'reg_lambda': 1, 'n_estimators': 400}
0.7843  {'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'reg_lambda': 0, 'n_estimators': 400}
0.7774  {'learning_

In [218]:
PARAMS = {**PARAMS_HAND, **results[0][1]}
print(PARAMS)

{'objective': 'binary', 'learning_rate': 0.03, 'num_leaves': 63, 'min_child_samples': 20, 'feature_fraction': 0.8, 'bagging_fraction': 0.9, 'bagging_freq': 1, 'n_estimators': 400, 'verbose': -1, 'n_jobs': -1, 'reg_lambda': 0}


### 7.4 Сколько из выигрыша настоящее

Лучшая конфигурация обошла ручную. Но если взять 12 одинаковых по сути
конфигураций и померить каждую, лучшая всё равно окажется выше средней:
в каждом замере есть случайная составляющая, и максимум из 12 её
подбирает

Посчитаем, сколько это даёт. Разыгрываем шум с нашей сигмой 12 раз, берём
максимум, повторяем 20 тысяч раз. Среднее покажет, какой выигрыш
получается от одного только отбора

In [226]:
sim = np.random.RandomState(1).normal(0, SIGMA, size=(20000, len(results))).max(axis=1)
expected_luck = sim.mean()
observed_gain = results[0][0] - pr_hand

print(f"ожидаемый выигрыш от отбора лучшего из {len(results)}: {expected_luck:+.4f}")
print(f"наблюдаемый выигрыш лучшей конфигурации:     {observed_gain:+.4f}")
print(f"разброс всех конфигураций: {results[-1][0]:.4f} - {results[0][0]:.4f}")

ожидаемый выигрыш от отбора лучшего из 12: +0.0022
наблюдаемый выигрыш лучшей конфигурации:     +0.0010
разброс всех конфигураций: 0.7651 - 0.7843


Выигрыш лучшей конфигурации не больше того, что даёт сам отбор. Значит
разница между конфигурациями в пределах погрешности

Лучшую всё равно берём: она не хуже ручной и втрое быстрее за счёт
меньшего числа деревьев

## 8. Качество итоговой модели

In [227]:
oof = cv_oof(Xtr[cols], y, params=PARAMS)
report("CV, 5 сидов", oof)

CV, 5 сидов    P@R70 0.7925 | PR 0.7856 | ROC 0.9203 | R@FPR1% 0.6641


При 8% позитивов отрицательный класс большой и легко ранжируется, поэтому высокое значение получит и
посредственная модель. Смотрим на PR-AUC и на целевую метрику.

R@FPR1% читается так: если задевать не больше 1% живых пользователей,
поймаем около 66% ботов

In [228]:
# теперь по временному сплиту:
ts = np.mean([lgb.LGBMClassifier(**PARAMS, random_state=s)
              .fit(Xtr[cols][~is_valid], y[~is_valid])
              .predict_proba(Xtr[cols][is_valid])[:, 1] for s in SEEDS], axis=0)
print('time-split P@R70:', round(precision_at_recall(y[is_valid], ts), 4))

time-split P@R70: 0.7943


Временной сплит дал 0.7943 против 0.7925 на кросс-валидации. Числа сошлись, значит модель, обученная на первых 2 неделях
апреля, одинаково хорошо работает и на перемешанных фолдах, и на последних
трёх днях

Если бы временной сплит заметно просел, это скорее всего означало бы, что какой-то
признак завязан на конкретный период и на тесте сломается

## 9. Финальное предикт и сабмит

Здесь обучаемся на всём трейне (кросс-валидация нужна была только для
измерения), усреднение по пяти сидам

In [229]:
assert list(Xtr.columns) == list(Xte.columns), "разный набор или порядок колонок"
assert (Xte.cookie_id.values == test.cookie_id.values).all()

score = np.zeros(len(test))
for sd in SEEDS:
    m = lgb.LGBMClassifier(**PARAMS, random_state=sd).fit(Xtr[cols], y)
    score += m.predict_proba(Xte[cols])[:, 1]
score /= len(SEEDS)

sub = pd.DataFrame({"cookie_id": test.cookie_id, "score": score})

assert len(sub) == len(test)
assert sub.cookie_id.duplicated().sum() == 0
assert set(sub.cookie_id) == set(test.cookie_id)
assert sub.score.notna().all() and sub.score.between(0, 1).all()
assert list(sub.columns) == ['cookie_id', 'score']

sub.to_csv("submission.csv", index=False)
print(sub.shape, "| уникальных значений score:", sub.score.nunique())
print(sub.score.describe())

(4909, 2) | уникальных значений score: 4909
count    4909.000000
mean        0.065667
std         0.227441
min         0.000007
25%         0.000253
50%         0.000719
75%         0.002862
max         0.999914
Name: score, dtype: float64


In [8]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import sklearn

reqs = "\n".join([
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"lightgbm=={lgb.__version__}",
]) + "\n"

with open("requirements.txt", "w") as f:
    f.write(reqs)
print(reqs)

pandas==3.0.6
numpy==2.5.3
scikit-learn==1.9.1
lightgbm==4.7.0

